# Phase 1.3: `probes/cv.py`, the leakage-safe split definition (tile 32UNU)

**Any number produced outside `probes/cv.py` does not exist.** This notebook is
where that module is exercised end to end: every runnable mode, every refusal,
the same-cube gate provoked on purpose, and the join contract to the Phase 1.2
embeddings asserted on real (cube, encoder) pairs.

**CPU is enough.** No GPU, no encoder weights, <= 12 GB. Nothing is fine-tuned
here and nothing is re-encoded.

**What must RAISE on this subset, and is not a bug.** The 20 cubes are one tile
(32UNU) and one year (2018). So `cube`, `spatial_block` and `temporal` run,
while `year`, `tile` and `crossed` all correctly refuse. Step 8 asserts the
refusals; a green Step 8 means the guards work.

**Drive layout: one SUBFOLDER per phase, under one project folder.** Deleting
a phase's subfolder removes everything that phase created and nothing another
phase depends on:

```
My Drive/
└── NeurIPS-CCAI-2026/
    ├── data/raw/*.nc         SHARED cubes. NOT a phase -- every phase reads
    │                         them, and reset_phase refuses to touch them.
    ├── phase1_1/             checkout + notebook
    ├── phase1_2/             checkout; artefacts at data/phase1_2/{embeddings,masks}
    └── phase1_3/             checkout
        └── phase1_3_repo.zip <- drag it here, leave it zipped
```

Step 2 finds the cubes and the Phase 1.2 artefacts wherever they sit on Drive
(searching up to three levels down) and reads them IN PLACE. It never writes
into another phase's subfolder.

The `phase1_2/data/phase1_2/` nesting is redundant but deliberate: the outer
name is the Drive checkout, the inner one is `data.paths.phase_dir`, which is
canonical and not worth bending for cosmetics.

## Step 1: Install, then restart

CPU only. No `satlaspretrain-models` and no model weights: this phase reads the
`.npz` files Phase 1.2 already wrote.

In [1]:
import importlib.util, os, IPython
SENTINEL = "/content/.phase1_3_installed"
try:
    import google.colab            # noqa: F401
    ON_COLAB = True
except ImportError:
    # find_spec("google.colab") is NOT equivalent: it raises rather than
    # returning None when the parent `google` package is absent.
    ON_COLAB = False

if not ON_COLAB:
    print("not on Colab: skipping the install and the restart.")
    print("Run the notebook against your own environment (pip install -r "
          "requirements.txt) and continue from Step 2.")
elif os.path.exists(SENTINEL):
    print("Already installed in this runtime, skipping.")
    print(f"(delete {SENTINEL} and re-run to force a reinstall)")
else:
    # Not -q. A pip resolution failure here is the likeliest cause of every
    # later failure, and -q hides it.
    !pip install earthnet s3fs xarray zarr netCDF4

    # torch arrives with Colab and is imported transitively by encoders/.
    # Installing over Colab's build swaps in a slower wheel for no gain here.
    if importlib.util.find_spec("torch") is None:
        !pip install torch

    import subprocess, sys
    probe = "import s3fs, xarray, zarr, netCDF4, earthnet, pandas, numpy, torch"
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(
            "Install did not take. Read the pip output above for the real "
            "conflict. Do not continue: Step 6 would fail to build the manifest."
        )

    open(SENTINEL, "w").write("ok")
    print("\n" + "=" * 70)
    print("INSTALL VERIFIED. RESTARTING THE RUNTIME NOW. This is expected.")
    print("When it comes back, continue from Step 2. Do not re-run this cell.")
    print("=" * 70)
    IPython.get_ipython().kernel.do_shutdown(True)

not on Colab: skipping the install and the restart.
Run the notebook against your own environment (pip install -r requirements.txt) and continue from Step 2.


## Step 2: Bootstrap

Extracts `phase1_3_repo.zip` into **its own** `phase1_3/` subfolder, then
resolves two READ-ONLY inputs that live wherever they already are:

* `data/raw/` — the 20 cubes. Shared across phases and never cleared.
* `data/phase1_2/embeddings/` — the 100 `.npz` from Phase 1.2.

Everything Phase 1.3 writes goes under this subfolder's `data/phase1_3/`, so
`reset_phase("phase1_3")` (or deleting the `phase1_3/` subfolder) is a
complete undo.

In [2]:
import os, sys, glob, zipfile, textwrap

REQUIRED = ["data/ndvi.py", "data/loader.py", "data/paths.py",
            "data/climatology.py", "encoders/manifest.py",
            "encoders/pipeline.py", "probes/cv.py",
            "tests/test_cv_folds.py", "tests/conftest.py"]
ZIP_NAME = "phase1_3_repo.zip"
PHASE = "phase1_3"
INPUT_PHASE = "phase1_2"          # read-only: this phase never writes to it

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
except ImportError:
    DRIVE = None
    print("not on Colab, assuming the repo is the current directory")

def looks_like_repo(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in REQUIRED)

REPO = None
if DRIVE:
    zips = glob.glob(f"{DRIVE}/**/{ZIP_NAME}", recursive=True)
    unzipped = [os.path.dirname(os.path.dirname(h))
                for d in ("*", "*/*", "*/*/*")
                for h in glob.glob(f"{DRIVE}/{d}/probes/cv.py")]
    unzipped = [d for d in unzipped if looks_like_repo(d)]

    if zips:
        REPO = os.path.dirname(zips[0])
        marker = os.path.join(REPO, "probes", "cv.py")
        # Re-extract when the zip is newer than what is on disk. Without this a
        # freshly uploaded zip is ignored because an old checkout sits next to
        # it, and you debug last week's code.
        stale = (not os.path.exists(marker)
                 or os.path.getmtime(zips[0]) > os.path.getmtime(marker))
        if stale:
            print(f"found {zips[0]}")
            print(f"extracting into {REPO} (zip is newer)")
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(REPO)
        else:
            print(f"using existing checkout at {REPO} (zip is not newer)")
    elif unzipped:
        REPO = unzipped[0]
        print(f"found unzipped repo, no zip present: {REPO}")
else:
    # Off Colab, walk up from the working directory: running the notebook from
    # notebooks/ is normal and must not be mistaken for a missing checkout.
    d = os.getcwd()
    while not looks_like_repo(d) and os.path.dirname(d) != d:
        d = os.path.dirname(d)
    REPO = d

if not looks_like_repo(REPO):
    raise RuntimeError(textwrap.dedent(f"""
        Could not find the Phase 1.3 code.

        Fix, 2 minutes:
          1. Run make_zip.sh locally to build {ZIP_NAME}
          2. Open https://drive.google.com
          3. Make a NEW subfolder  My Drive / NeurIPS-CCAI-2026 / phase1_3
          4. Drag {ZIP_NAME} into it (do not unzip)
          5. Re-run this cell.

        One subfolder per phase is deliberate: deleting phase1_3/ removes
        everything Phase 1.3 created and nothing Phase 1.2 depends on.
        data/raw stays at the project root -- it is shared, not a phase.

        Searched under: {DRIVE}
        Needed all of: {REQUIRED}
        Resolved REPO = {REPO}
    """).strip())

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

from data.paths import RAW_DIR, describe_phase, phase_dir

# --- READ-ONLY inputs, resolved wherever they already live -----------------
# Phase 1.2 may have run in a different Drive folder. Read its artefacts in
# place; copying 70 MB of cubes per phase is waste, and writing into its
# folder would break the "delete this folder to undo this phase" property.
def _find(rel, minimum=1, pattern="*"):
    # This checkout first, then Drive up to THREE levels down. Three, because
    # phases are subfolders of one project folder: the Phase 1.2 embeddings sit
    # at  MyDrive / NeurIPS-CCAI-2026 / phase1_2 / data/phase1_2/embeddings,
    # which is two wildcards, while the shared cubes at
    # MyDrive / NeurIPS-CCAI-2026 / data/raw  are one.
    here = os.path.join(REPO, rel)
    cands = [here]
    if DRIVE:
        for depth in ("*", "*/*", "*/*/*"):
            cands += sorted(glob.glob(f"{DRIVE}/{depth}/{rel}"))
    for c in cands:
        if os.path.isdir(c) and len(glob.glob(os.path.join(c, pattern))) >= minimum:
            return c
    return here          # may not exist yet; the caller says what to do

RAW = _find(RAW_DIR, minimum=1, pattern="*.nc")
EMB_IN = _find(os.path.join("data", INPUT_PHASE, "embeddings"), minimum=1,
               pattern="*.npz")
os.makedirs(RAW, exist_ok=True)

# --- this phase's OWN outputs ----------------------------------------------
FOLDS = phase_dir(PHASE, "folds")

n_cubes = len(glob.glob(os.path.join(RAW, "*.nc")))
n_emb = len(glob.glob(os.path.join(EMB_IN, "*.npz")))
print(f"\nREPO    {REPO}")
print(f"RAW     {RAW}   ({n_cubes} cubes)"
      + ("" if n_cubes else "   <- Step 4 downloads them"))
print(f"EMB_IN  {EMB_IN}   ({n_emb} .npz, READ-ONLY)"
      + ("" if n_emb else "   <- MISSING, see Step 3"))
print(f"FOLDS   {FOLDS}   (this phase writes here only)")
describe_phase(PHASE)

from data.ndvi import ndvi
from encoders.manifest import build_manifest
from probes import cv
print(f"\nimports OK. canonical NDVI at {ndvi.__module__}, "
      f"splits at {cv.__name__}, modes {cv.MODES}")
for f in REQUIRED:
    print(f"  ok  {f}")


# --- shell helper, defined here so it can never be skipped ------------------
# Named sh(), not run(): IPython has a %run magic. If a helper called run() is
# ever undefined, automagic silently rewrites run("...") into %run("...") and
# reports a confusing error about a missing script instead of a NameError.
import shlex, subprocess

# The interpreter that imported this repo, not whatever `python` resolves to.
# On a pyenv/venv machine a bare `python` may not exist at all, and on Colab it
# may not be the kernel's interpreter -- either way a subprocess would then
# test different code than the notebook is holding.
PY = shlex.quote(sys.executable)

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")
    print(f"[exit 0] {cmd}")

print("helper ready: sh('<shell command>')")

not on Colab, assuming the repo is the current directory

REPO    /Users/benji/Code/NeurIPS CCAI 2026
RAW     /Users/benji/Code/NeurIPS CCAI 2026/data/raw   (20 cubes)
EMB_IN  /Users/benji/Code/NeurIPS CCAI 2026/data/phase1_2/embeddings   (80 .npz, READ-ONLY)
FOLDS   data/phase1_3/folds   (this phase writes here only)
[paths] data/phase1_3: 5 file(s), 0.20 MB
[paths]   folds/: 5 file(s), 0.20 MB



imports OK. canonical NDVI at data.ndvi, splits at probes.cv, modes ('cube', 'crossed', 'year', 'tile', 'spatial_block', 'temporal')
  ok  data/ndvi.py
  ok  data/loader.py
  ok  data/paths.py
  ok  data/climatology.py
  ok  encoders/manifest.py
  ok  encoders/pipeline.py
  ok  probes/cv.py
  ok  tests/test_cv_folds.py
  ok  tests/conftest.py
helper ready: sh('<shell command>')


## Step 3: Environment check

Phase 1.3 needs no GPU and no encoder weights. It does need the Phase 1.2
embeddings, because Step 10 asserts the join contract against them.

In [3]:
import glob, os, shutil
import numpy as np, pandas as pd

print("numpy      ", np.__version__)
print("pandas     ", pd.__version__)
try:
    import torch
    print("torch      ", torch.__version__,
          "| CPU is sufficient for this phase; no weights are loaded")
except ImportError:
    raise RuntimeError("torch missing: encoders/ imports it transitively. Re-run Step 1.")

n_emb = len(glob.glob(os.path.join(EMB_IN, "*.npz")))
print(f"\ncubes      {len(glob.glob(os.path.join(RAW, '*.nc')))} in {RAW}")
print(f"embeddings {n_emb} in {EMB_IN}")
if n_emb == 0:
    raise RuntimeError(textwrap.dedent(f"""
        No Phase 1.2 embeddings found. Step 10 cannot assert the join contract
        without them.

        Fix: run notebooks/phase1_2_encoders.ipynb (its own Drive folder is
        fine -- Step 2 searches Drive for data/{INPUT_PHASE}/embeddings and
        reads it in place), then re-run Step 2 here.

        Searched from: {REPO}  and one/two levels under {DRIVE}
    """).strip())

free = shutil.disk_usage(REPO).free / 1e9
print(f"free space {free:.1f} GB (this phase writes a few hundred kB of fold indices)")
assert free > 0.5, "less than 0.5 GB free, clear space on Drive first"

numpy       2.0.2
pandas      2.3.3
torch       2.8.0 | CPU is sufficient for this phase; no weights are loaded

cubes      20 in /Users/benji/Code/NeurIPS CCAI 2026/data/raw
embeddings 80 in /Users/benji/Code/NeurIPS CCAI 2026/data/phase1_2/embeddings
free space 30.1 GB (this phase writes a few hundred kB of fold indices)


## Step 4: The cubes

The manifest is built from `data/raw/*.nc`, **not** from the `.npz` files.
Already-present cubes are skipped.

In [4]:
if len(glob.glob(os.path.join(RAW, "*.nc"))) >= 20:
    print(f"20 cubes already in {RAW}, skipping the download")
else:
    sh(f"{PY} -m data.download_greenearthnet --out '{RAW}' --n 20 --tile 32UNU")
print(f"{len(glob.glob(os.path.join(RAW, '*.nc')))} cubes in {RAW}")

20 cubes already in /Users/benji/Code/NeurIPS CCAI 2026/data/raw, skipping the download
20 cubes in /Users/benji/Code/NeurIPS CCAI 2026/data/raw


## Step 5: Unit tests

Expect **`146 passed, 5 skipped`** — 118 from Phases 1.1/1.2 plus 28 new fold
tests. The 5 skips are the weight-downloading and MI batch-invariance tests,
gated behind `PHASE1_2_WEIGHTS=1`.

A **different collected count** than you get locally means the bundle is stale:
`make_zip.sh` lists files with `git ls-files`, so an uncommitted file is
silently absent. Commit, rebuild, re-upload. That is a signal, not noise.

In [5]:
sh(f"{PY} -m pytest tests -q")

$ '/Users/benji/Code/NeurIPS CCAI 2026/.venv/bin/python3' -m pytest tests -q


........................................................................ [ 47%]


ssss...........s........................................................ [ 95%]
.......                                                                  [100%]
=============================== warnings summary ===============================
tests/test_encoders.py::test_grid_landcover_aligns_with_the_embedding_grid
  <frozen importlib._bootstrap>:228: RuntimeWarning: numpy.ndarray size changed, may indicate binary incompatibility. Expected 16 from C header, got 96 from PyObject

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html


[exit 0] '/Users/benji/Code/NeurIPS CCAI 2026/.venv/bin/python3' -m pytest tests -q


## Step 6: The REAL manifest, built from the cubes

`encoders.manifest.build_manifest` already exists — it is imported, never
rebuilt. One row per RETAINED (cube, frame).

In [6]:
import glob
from data.loader import load_cube
from encoders.manifest import assert_strata_present, build_manifest

paths = sorted(glob.glob(os.path.join(RAW, "*.nc")))
assert len(paths) == 20, f"expected 20 cubes, found {len(paths)}"
samples = [load_cube(p, verbose=False) for p in paths]
MANIFEST = build_manifest(samples)
assert_strata_present(MANIFEST)

print(f"\nmanifest {MANIFEST.shape}   rows x columns")
print(f"cubes  {MANIFEST.cube_id.nunique()}   tiles {sorted(MANIFEST.tile.unique())}   "
      f"years {sorted(MANIFEST.year.unique())}")
print(f"clear_frac  min {MANIFEST.clear_frac.min():.3f}  "
      f"median {MANIFEST.clear_frac.median():.3f}  max {MANIFEST.clear_frac.max():.3f}")
print(f"timestamps  {str(MANIFEST.timestamp.min())[:10]} .. "
      f"{str(MANIFEST.timestamp.max())[:10]}")
assert len(MANIFEST) == 264, f"expected 264 retained frames, got {len(MANIFEST)}"
print("\nONE TILE, ONE YEAR -- so year / tile / crossed MUST refuse (Step 8).")

[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 122/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 122/150 timesteps with no acquisition


...

[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 120/150 timesteps with no acquisition


[loader] dropping 120/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


[loader] dropping 121/150 timesteps with no acquisition
[loader] dropping 121/150 timesteps with no acquisition


[manifest] 264 (cube, frame) rows over 20 cubes
[manifest] columns: ['cube_id', 'tile', 'year', 'timestamp', 'original_axis_index', 'day_of_year', 'pixel_bbox', 'clear_frac', 'landcover_stratum', 'landcover_dominant_frac', 'grid_landcover', 'grid_landcover_purity', 'grid_elevation_m', 'eobs_tg', 'eobs_fg', 'eobs_hu', 'eobs_pp', 'eobs_qq', 'eobs_rr', 'eobs_tn', 'eobs_tx']
[manifest] landcover strata (per cube): {'cropland': 8, 'grassland': 6, 'tree_cover': 6}
[manifest] landcover strata (per grid cell, 320 cells over 20 cubes): {'cropland': 127, 'tree_cover': 99, 'grassland': 88, 'built_up': 5, 'bare_sparse': 1}
[manifest] cubes whose cells are NOT all one class: 19/20 -- this is the within-cube stratum contrast the per-cube label was hiding
[manifest] in-cube E-OBS joined on original_axis_index (8): ['eobs_fg', 'eobs_hu', 'eobs_pp', 'eobs_qq', 'eobs_rr', 'eobs_tg', 'eobs_tn', 'eobs_tx']
[manifest] original_axis_index spans 0..29 (range 29); horizons are defined in DAYS on this axis

ma

## Step 7: The three runnable modes

`cube` (default), `spatial_block` and `temporal`. Each prints, per fold:
n_train rows, n_test rows, n_train cubes, n_test cubes, and the years on each
side. The clear-fraction filter runs before every split.

In [7]:
from probes import cv

print("=" * 70, "\ncube, k=5   (DEFAULT: GroupKFold on cube_id)\n" + "=" * 70)
cube5 = list(cv.folds(MANIFEST, "cube", k=5))

print("\n" + "=" * 70)
print("leave-one-cube-out   (the honest choice at 20-cube scale)\n" + "=" * 70)
loco = list(cv.leave_one_cube_out(MANIFEST, verbose=False))
print(f"[cv] {len(loco)} folds, each holding out exactly one cube; "
      f"test sizes {sorted(te.size for _, te in loco)}")

print("\n" + "=" * 70)
print("spatial_block, k=5   (SUBSTITUTE for tile holdout)\n" + "=" * 70)
blocks = list(cv.folds(MANIFEST, "spatial_block", k=5))

print("\n" + "=" * 70)
print("temporal, cutoff 2018-08-15   (P3 robustness variant, NOT a default)\n"
      + "=" * 70)
temporal = list(cv.folds(MANIFEST, "temporal", cutoff="2018-08-15"))

# Independent re-check of the leakage rule, computed here rather than trusted.
for name, fs in [("cube", cube5), ("loco", loco), ("spatial_block", blocks),
                 ("temporal", temporal)]:
    for i, (tr, te) in enumerate(fs):
        a = set(MANIFEST.cube_id.to_numpy()[tr])
        b = set(MANIFEST.cube_id.to_numpy()[te])
        assert not (a & b), f"{name} fold {i}: cube on both sides"
        assert not np.intersect1d(tr, te).size
print(f"\nRE-CHECKED independently: no cube on both sides in any of "
      f"{len(cube5) + len(loco) + len(blocks) + len(temporal)} folds")

cube, k=5   (DEFAULT: GroupKFold on cube_id)
[cv] clear_frac filter (> 0.5): kept 264/264 rows, removed 0
[cv] cube fold 1/5: train 211 rows / 16 cubes, test 53 rows / 4 cubes | years train [2018] test [2018]
[cv] cube fold 2/5: train 212 rows / 16 cubes, test 52 rows / 4 cubes | years train [2018] test [2018]
[cv] cube fold 3/5: train 211 rows / 16 cubes, test 53 rows / 4 cubes | years train [2018] test [2018]
[cv] cube fold 4/5: train 211 rows / 16 cubes, test 53 rows / 4 cubes | years train [2018] test [2018]
[cv] cube fold 5/5: train 211 rows / 16 cubes, test 53 rows / 4 cubes | years train [2018] test [2018]

leave-one-cube-out   (the honest choice at 20-cube scale)
[cv] 20 folds, each holding out exactly one cube; test sizes [10, 11, 12, 12, 12, 12, 13, 13, 13, 13, 13, 13, 14, 14, 14, 14, 15, 15, 15, 16]

spatial_block, k=5   (SUBSTITUTE for tile holdout)
[cv] clear_frac filter (> 0.5): kept 264/264 rows, removed 0
[cv] spatial_block: 1 tile, 20 cubes clustered by pixel_bbox into

## Step 8: The three refusals

`year`, `tile` and `crossed` must RAISE here. A raise is the designed
behaviour on a single-tile, single-year subset — read the messages: each one
names the correct fallback and warns against a random split.

In [8]:
expected = [
    ("year", lambda: list(cv.folds(MANIFEST, "year")), cv.SingleYearError),
    ("tile", lambda: list(cv.folds(MANIFEST, "tile")), cv.SingleTileError),
    ("crossed", lambda: list(cv.folds(MANIFEST, "crossed")), cv.SingleYearError),
]
for name, call, err in expected:
    try:
        call()
        raise AssertionError(f"{name} mode did NOT raise -- the guard is broken")
    except err as e:
        print(f"--- {name} mode raised {type(e).__name__}, correctly ---")
        print(str(e), "\n")
print("all three refusals fired")

[cv] clear_frac filter (> 0.5): kept 264/264 rows, removed 0
--- year mode raised SingleYearError, correctly ---
year-grouped CV needs at least 2 years, the manifest has [2018] (20 cubes). Every cube in tile 32UNU is from 2018 (the extreme split), so this is expected on the current subset. Use 'cube' mode (spatial grouping) here, or 'crossed' once the manifest spans years. Do not fall back to a random split: it would put the same season on both sides. 

[cv] clear_frac filter (> 0.5): kept 264/264 rows, removed 0
--- tile mode raised SingleTileError, correctly ---
tile-grouped CV needs at least 2 tiles, the manifest has ['32UNU']. The 20-cube prototype is tile 32UNU only, so this is expected: use 'spatial_block' as the prototype-scale substitute (whole within-tile clusters held out). True tile holdout is DEFERRED TO SCALE-UP. Do not fall back to a random split. 

[cv] clear_frac filter (> 0.5): kept 264/264 rows, removed 0
--- crossed mode raised SingleYearError, correctly ---
crossed 

## Step 9: The same-cube gate, provoked on purpose

The gate lives INSIDE the splitter, not in the caller. Two provocations on
small synthetic manifests:

1. A **seasonal-shaped** manifest where one cube spans 2018–2020. Year grouping
   must then split within a cube, so `year` mode refuses and names `crossed`.
   `crossed` handles the same manifest cleanly — that is the collision resolved.
2. A manifest with a **duplicated `(cube_id, timestamp)` row**, which must fail
   loudly rather than duplicate a frame across folds.

In [9]:
def synthetic(cubes, years, frames=4):
    rows = []
    for c in cubes:
        i = 0
        for y in years:
            for f in range(frames):
                rows.append({"cube_id": c, "tile": "33TAN", "year": y,
                             "timestamp": np.datetime64(f"{y}-05-01")
                                          + np.timedelta64(10 * f, "D"),
                             "original_axis_index": i,
                             "pixel_bbox": (0, 128, 0, 128), "clear_frac": 0.8})
                i += 1
    return pd.DataFrame(rows)

seasonal = synthetic([f"S{c}.nc" for c in range(4)], [2018, 2019, 2020])
print(f"synthetic seasonal manifest: {len(seasonal)} rows, "
      f"{seasonal.cube_id.nunique()} cubes each spanning "
      f"{sorted(seasonal.year.unique())}\n")

try:
    list(cv.year_folds(seasonal, k=3, verbose=False))
    raise AssertionError("the same-cube gate did NOT fire")
except cv.LeakageError as e:
    print("--- year mode on a multi-year cube raised LeakageError, correctly ---")
    print(str(e), "\n")

print("--- crossed mode on the SAME manifest, which is the resolution ---")
for tr, te in cv.crossed_folds(seasonal, k=3):
    ts = np.asarray(seasonal.timestamp.to_numpy(), dtype="datetime64[ns]")
    yr = ts.astype("datetime64[Y]").astype(int) + 1970
    assert not (set(seasonal.cube_id.to_numpy()[tr]) & set(seasonal.cube_id.to_numpy()[te]))
    assert not (set(yr[tr]) & set(yr[te]))
print("crossed: every test row's cube AND year unseen in train (asserted)\n")

dup = pd.concat([seasonal, seasonal.iloc[[0]]], ignore_index=True)
try:
    list(cv.cube_folds(dup, verbose=False))
    raise AssertionError("duplicate rows did NOT fail")
except cv.LeakageError as e:
    print("--- a duplicated (cube_id, timestamp) row raised LeakageError ---")
    print(str(e))

synthetic seasonal manifest: 48 rows, 4 cubes each spanning [np.int64(2018), np.int64(2019), np.int64(2020)]

--- year mode on a multi-year cube raised LeakageError, correctly ---
year fold 1/3 puts frames of 4 cube(s) on BOTH sides, e.g. 'S0.nc'. Frames of one cube must never be split across a fold, whatever mode asked for it. A cube spanning a year boundary cannot be split by year (the seasonal-split collision); use 'crossed' mode, which holds cube and year out jointly. 

--- crossed mode on the SAME manifest, which is the resolution ---
[cv] clear_frac filter (> 0.5): kept 48/48 rows, removed 0
[cv] crossed fold 1/3: train 16 rows / 2 cubes, test 8 rows / 2 cubes | years train [2019, 2020] test [2018]
[cv] crossed fold 2/3: train 24 rows / 3 cubes, test 4 rows / 1 cubes | years train [2018, 2020] test [2019]
[cv] crossed fold 3/3: train 24 rows / 3 cubes, test 4 rows / 1 cubes | years train [2018, 2019] test [2020]
crossed: every test row's cube AND year unseen in train (asserted)



## Step 10: The join contract, on real (cube, encoder) pairs

    (cube_id, original_axis_index)  ==  (cube, kept_idx)

`join_embeddings` asserts it on every shared field — kept_idx, timestamps,
clear_frac — and carries `window_span_days` through. The multi-image encoder's
lookback varies over ~0–105 days and correlates with cloud, so it is a
covariate a probe must be able to condition on; the join REFUSES to drop it.

In [10]:
from encoders.pipeline import load_encoded

encoders_present = sorted({os.path.basename(p).split("__")[1][:-4]
                           for p in glob.glob(os.path.join(EMB_IN, "*.npz"))})
print(f"encoders in {EMB_IN}: {encoders_present}\n")

joined = {}
for enc in encoders_present:
    p = sorted(glob.glob(os.path.join(EMB_IN, f"*__{enc}.npz")))[0]
    out = cv.join_embeddings(MANIFEST, load_encoded(p))
    joined[enc] = out
    rows = out["manifest_idx"]
    # Re-assert the contract here too, independently of the function.
    assert (MANIFEST.original_axis_index.to_numpy()[rows]
            == load_encoded(p).kept_idx).all()
    assert out["window_span_days"].shape[0] == rows.size

print(f"\n{len(joined)} (cube, encoder) pairs joined, contract asserted on each")
mi = [e for e in joined if e.endswith("_mi_rgb")]
if mi:
    w = joined[mi[0]]["window_span_days"]
    print(f"window_span_days carried through for {mi[0]}: "
          f"min {w.min():.0f}  median {np.median(w):.0f}  max {w.max():.0f} days "
          "-- the covariate P2/P3 must condition on")
si = [e for e in joined if not e.endswith("_mi_rgb")]
if si:
    assert all((joined[e]["window_span_days"] == 0).all() for e in si)
    print(f"single-image encoders {si}: window_span_days == 0 exactly "
          "(an honest constant, not a NaN)")

encoders in /Users/benji/Code/NeurIPS CCAI 2026/data/phase1_2/embeddings: ['imagenet_vit_b16', 'raw_features', 'satlas_s2_swinb_mi_rgb', 'satlas_s2_swinb_rgb']

[cv] join 32UNU_2018-03-09_2018-08-05_1081_1209_3641_3769_16_96_56_136.nc x imagenet_vit_b16: manifest_idx (14,) | embeddings (14, 1536) | grid (14, 16, 768) | window_span_days (14,) (min 0 max 0 days)
[cv] join 32UNU_2018-03-09_2018-08-05_1081_1209_3641_3769_16_96_56_136.nc x raw_features: manifest_idx (14,) | embeddings (14, 35) | grid (14, 16, 35) | window_span_days (14,) (min 0 max 0 days)
[cv] join 32UNU_2018-03-09_2018-08-05_1081_1209_3641_3769_16_96_56_136.nc x satlas_s2_swinb_mi_rgb: manifest_idx (14,) | embeddings (14, 1024) | grid (14, 16, 1024) | window_span_days (14,) (min 0 max 85 days)
[cv] join 32UNU_2018-03-09_2018-08-05_1081_1209_3641_3769_16_96_56_136.nc x satlas_s2_swinb_rgb: manifest_idx (14,) | embeddings (14, 1024) | grid (14, 16, 1024) | window_span_days (14,) (min 0 max 0 days)

4 (cube, encoder) pairs j

## Step 11: Save this phase's fold indices

Written through `data/paths.phase_dir`, never a hand-typed path. Everything
lands under this folder's `data/phase1_3/`, so `reset_phase("phase1_3")` — or
deleting the Drive folder — is a complete, phase-scoped undo.

In [11]:
from data.paths import describe_phase, phase_dir

FOLDS = phase_dir(PHASE, "folds")
saved = []
for name, fs in [("cube_k5", cube5), ("loco", loco),
                 ("spatial_block_k5", blocks), ("temporal_2018-08-15", temporal)]:
    path = os.path.join(FOLDS, f"{name}.npz")
    np.savez_compressed(
        path,
        n_folds=np.array(len(fs)),
        n_manifest_rows=np.array(len(MANIFEST)),
        **{f"train_{i}": tr for i, (tr, _) in enumerate(fs)},
        **{f"test_{i}": te for i, (_, te) in enumerate(fs)},
    )
    saved.append(path)
    print(f"[phase1_3] {os.path.basename(path)}: {len(fs)} folds, "
          f"{os.path.getsize(path) / 1e3:.1f} kB")

# The manifest itself, so a probe can reproduce a fold without re-reading cubes.
mpath = os.path.join(FOLDS, "manifest.csv")
MANIFEST.to_csv(mpath, index=False)
print(f"[phase1_3] manifest.csv: {MANIFEST.shape} "
      f"({os.path.getsize(mpath) / 1e3:.0f} kB)")

# Read one back and re-assert it, rather than trusting the write.
z = np.load(saved[0])
assert int(z["n_manifest_rows"]) == len(MANIFEST)
tr0, te0 = z["train_0"], z["test_0"]
assert not set(MANIFEST.cube_id.to_numpy()[tr0]) & set(MANIFEST.cube_id.to_numpy()[te0])
print(f"\nre-read {os.path.basename(saved[0])}: fold 0 train {tr0.shape} "
      f"test {te0.shape}, still cube-disjoint")
describe_phase(PHASE)

[phase1_3] cube_k5.npz: 5 folds, 4.7 kB
[phase1_3] loco.npz: 20 folds, 17.6 kB
[phase1_3] spatial_block_k5.npz: 5 folds, 4.7 kB
[phase1_3] temporal_2018-08-15.npz: 1 folds, 1.2 kB
[phase1_3] manifest.csv: (264, 21) (172 kB)

re-read cube_k5.npz: fold 0 train (211,) test (53,), still cube-disjoint
[paths] data/phase1_3: 5 file(s), 0.20 MB
[paths]   folds/: 5 file(s), 0.20 MB


{'phase': 'phase1_3',
 'dir': 'data/phase1_3',
 'files': 5,
 'bytes': 200318,
 'by_kind': {'folds': {'files': 5, 'bytes': 200318}}}

## Phase 1.3 is done when

```
Step 5   146 passed, 5 skipped
Step 6   manifest (264, 21), 20 cubes, tile ['32UNU'], years [2018]
Step 7   cube k=5, LOCO (20 folds), spatial_block k=5, temporal all yield folds
         and the independent re-check finds no cube on both sides
Step 8   year -> SingleYearError, tile -> SingleTileError,
         crossed -> SingleYearError, each naming its fallback
Step 9   year mode on a multi-year cube -> LeakageError naming crossed;
         crossed handles the same manifest; a duplicate row -> LeakageError
Step 10  join contract asserted on one real pair per encoder,
         window_span_days carried through (0 for single-image, 0-105 for MI)
Step 11  fold indices + manifest under data/phase1_3/folds/
```

**Everything this phase wrote is under `data/phase1_3/`.** To re-run cleanly:

```python
from data.paths import reset_phase
reset_phase("phase1_3")     # clears ONLY this phase; data/raw is untouched
```

Deleting the `phase1_3/` subfolder is the coarser version of the same undo,
and it cannot touch Phase 1.2's artefacts or the shared cubes, because this
notebook only ever READS them.

Next: P1/P2/P3/P4 import `probes.cv.folds` and nothing else. Any number that
does not come out of a fold from this module does not exist.